In [ ]:
# --- repo bootstrap ---
import sys
from pathlib import Path

repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent

if not (repo / "src").exists():
    !git clone https://github.com/thinkthoughts/ion-transport-waveform-pipeline.git
    %cd ion-transport-waveform-pipeline
    repo = Path.cwd()

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

print("Repo root:", repo)


# 05 — Excitation Metrics

Formalize the excitation metrics used to evaluate ion transport.

```text
target path x_c(t) + ion trajectory x(t) → residual motion → amplitude / energy / spectrum
```

Notebook 04 showed a first residual-excitation estimate. This notebook separates the **metric layer** from the broader tradeoff analysis that comes next.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.ion_transport_waveform.config import TrapConfig, IonSpecies
from src.ion_transport_waveform.transport_path import minimum_jerk_path, linear_path
from src.ion_transport_waveform.motion_sim import simulate_ion_motion
from src.ion_transport_waveform.excitation_metrics import residual_amplitude, residual_energy_proxy

cfg = TrapConfig()
ion = IonSpecies()

fig_dir = repo / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

data_dir = repo / "data" / "simulation_outputs"
data_dir.mkdir(parents=True, exist_ok=True)

print("Figure dir:", fig_dir)
print("Data dir:", data_dir)


## 1. Define fixed transport case

Use one fixed transport task so this notebook focuses on metric definitions rather than parameter sweeps.

In [ ]:
x0 = -160e-6
x1 = 160e-6
duration = 20e-6
n_steps = 1600

t = np.linspace(0, duration, n_steps)
path_mj = minimum_jerk_path(t, x0=x0, x1=x1, duration=duration)
path_lin = linear_path(t, x0=x0, x1=x1, duration=duration)

x_mj, v_mj = simulate_ion_motion(t, path_mj, cfg.omega_rad_s)
x_lin, v_lin = simulate_ion_motion(t, path_lin, cfg.omega_rad_s)

res_mj = x_mj - path_mj
res_lin = x_lin - path_lin

print("minimum-jerk residual amplitude:", residual_amplitude(x_mj, path_mj))
print("linear residual amplitude:", residual_amplitude(x_lin, path_lin))


## 2. Residual motion time series

The residual coordinate is

```text
r(t) = x(t) - x_c(t)
```

This measures how far the ion lags or oscillates relative to the moving well center.

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(t * 1e6, res_mj * 1e9, label="minimum-jerk residual")
plt.plot(t * 1e6, res_lin * 1e9, label="linear residual", alpha=0.8)
plt.xlabel("time (µs)")
plt.ylabel("residual position r(t) (nm)")
plt.title("Residual ion motion relative to target path")
plt.legend()
plt.tight_layout()
plt.savefig(fig_dir / "05_excitation_residual_time_series.png", dpi=180)
plt.show()

print(f"saved: {fig_dir / '05_excitation_residual_time_series.png'}")


## 3. Time-resolved energy proxy

A classical harmonic residual-energy proxy is

```text
E_res(t) = 1/2 m [ v_rel(t)^2 + ω² r(t)^2 ]
```

where `r(t)` is residual displacement and `v_rel(t)` is residual velocity.

In [ ]:
dt = t[1] - t[0]
vrel_mj = np.gradient(res_mj, dt)
vrel_lin = np.gradient(res_lin, dt)

E_mj = 0.5 * ion.mass_kg * (vrel_mj**2 + (cfg.omega_rad_s * res_mj)**2)
E_lin = 0.5 * ion.mass_kg * (vrel_lin**2 + (cfg.omega_rad_s * res_lin)**2)

print("final E minimum-jerk:", E_mj[-1])
print("final E linear:", E_lin[-1])


In [ ]:
plt.figure(figsize=(8, 4.5))
plt.semilogy(t * 1e6, E_mj, label="minimum-jerk")
plt.semilogy(t * 1e6, E_lin, label="linear", alpha=0.8)
plt.xlabel("time (µs)")
plt.ylabel("residual energy proxy (J)")
plt.title("Time-resolved residual excitation energy")
plt.legend()
plt.tight_layout()
plt.savefig(fig_dir / "05_excitation_energy_time_series.png", dpi=180)
plt.show()

print(f"saved: {fig_dir / '05_excitation_energy_time_series.png'}")


## 4. Frequency-domain residual spectrum

The residual spectrum identifies which motional frequencies are excited by a transport path.

In [ ]:
def residual_spectrum(residual, dt):
    signal = residual - np.mean(residual)
    freqs = np.fft.rfftfreq(signal.size, d=dt)
    amp = np.abs(np.fft.rfft(signal))
    return freqs, amp

freq_mj, spec_mj = residual_spectrum(res_mj, dt)
freq_lin, spec_lin = residual_spectrum(res_lin, dt)

# avoid plotting DC-dominated first bin
mask = freq_mj > 0

print("trap secular frequency (MHz):", cfg.omega_rad_s / (2*np.pi) / 1e6)


In [ ]:
plt.figure(figsize=(8, 4.5))
plt.semilogy(freq_mj[mask] / 1e6, spec_mj[mask], label="minimum-jerk")
plt.semilogy(freq_lin[mask] / 1e6, spec_lin[mask], label="linear", alpha=0.8)
plt.axvline(cfg.omega_rad_s / (2*np.pi) / 1e6, linestyle="--", linewidth=1.0, alpha=0.6, label="secular freq")
plt.xlim(0, 5)
plt.xlabel("frequency (MHz)")
plt.ylabel("|FFT residual|")
plt.title("Residual-motion spectrum")
plt.legend()
plt.tight_layout()
plt.savefig(fig_dir / "05_excitation_fft_spectrum.png", dpi=180)
plt.show()

print(f"saved: {fig_dir / '05_excitation_fft_spectrum.png'}")


## 5. Compact metric table

This table gives a concise comparison between transport-path choices at the same duration.

In [ ]:
metrics = {
    "minimum_jerk": {
        "residual_amplitude_m": residual_amplitude(x_mj, path_mj),
        "final_energy_J": residual_energy_proxy(x_mj, v_mj, path_mj, cfg.omega_rad_s, mass_kg=ion.mass_kg),
        "peak_energy_J": float(np.max(E_mj)),
        "rms_residual_m": float(np.sqrt(np.mean(res_mj**2))),
    },
    "linear": {
        "residual_amplitude_m": residual_amplitude(x_lin, path_lin),
        "final_energy_J": residual_energy_proxy(x_lin, v_lin, path_lin, cfg.omega_rad_s, mass_kg=ion.mass_kg),
        "peak_energy_J": float(np.max(E_lin)),
        "rms_residual_m": float(np.sqrt(np.mean(res_lin**2))),
    },
}

for name, vals in metrics.items():
    print(name)
    for k, v in vals.items():
        print(f"  {k:24s} {v:.3e}")


## 6. Save metric outputs


In [ ]:
np.savez(
    data_dir / "excitation_metrics_05.npz",
    t=t,
    path_minimum_jerk=path_mj,
    path_linear=path_lin,
    x_minimum_jerk=x_mj,
    x_linear=x_lin,
    residual_minimum_jerk=res_mj,
    residual_linear=res_lin,
    energy_minimum_jerk=E_mj,
    energy_linear=E_lin,
    freq=freq_mj,
    spectrum_minimum_jerk=spec_mj,
    spectrum_linear=spec_lin,
)

print(f"saved: {data_dir / 'excitation_metrics_05.npz'}")


## 7. Next notebook

`06_tradeoff_analysis.ipynb` uses these metrics across parameter sweeps:

```text
duration / voltage / slew constraints → excitation tradeoff maps
```